<div align="right"><a href="S09-Sampling-and-Reconstruction.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

# Übung 2.10: Abtastung und Rekonstruktion

Ausgehend von einem zeitkontinuierlichen Signal $x_{\mathrm{K}}(t)$ und den
Eigenschaften der Fourier-Transformation (2.4.1, 2.4.2) erhalten wir eine
Folge $x(k)$, indem wir mit der Frequenz $f_A$ abtasten und das
Transformationspaar (2.3.11, 2.3.12) verwenden.  
Die Spektren $X_{\mathrm{K}}(j\omega)$ und $X(e^{j\Omega})$ sind durch den
Ausdruck (2.4.8) verknüpft:

$$
X(e^{j\Omega})=\frac{1}{T_A}\sum_{i=-\infty}^{\infty} X_{\mathrm{K}}\left(j\frac{\Omega+i2\pi}{T_A}\right)
$$

Somit kann das Spektrum eines diskreten Signals $X(e^{j\Omega})$  
durch Überlagerung der Spektren eines zeitkontinuierlichen Signals  
$X_{\mathrm{K}}(j\omega)$ erhalten werden, die um Vielfache von $2\pi$ verschoben  
und mit $1/T_A$ normiert sind.  

Um aus einem zeitdiskreten Signal wieder das zeitkontinuierliche Signal zu rekonstruieren,  
dürfen sich die Teilspektren $X_{\mathrm{K}}(j(\Omega+i2\pi)/T)$ nicht überlappen[^1].  
Dies führt zu den Anforderungen des sogenannten **Abtasttheorems** von Shannon:

[^1]: Der Fall, in dem sich diese Spektren überlappen, wird *Aliasing* genannt.

* $X_{\mathrm{K}}(j\omega)$ muss bandbegrenzt sein — es darf also im Bereich  
  $|\omega| \ge \omega_{\text{max}}$ verschwinden.
* Die Abtastfrequenz $\omega_A = 2\pi f_A = 2\pi /T_A$ muss mindestens doppelt so groß sein  
  wie die maximale Frequenz $\omega_{\text{max}}$ von $X_{\mathrm{K}}(j\omega)$.

---

Um Abtasteffekte zu untersuchen, ist es notwendig, das zeitkontinuierliche
Signal mit einer zeitdiskreten Folge zu simulieren.  
Dabei werden zwei Abtastperioden verwendet:

* Die Abtastperiode $\Delta t$ dient zur Simulation des kontinuierlichen Signals.
* Die Abtastperiode $T_A$ dient zur Abtastung des „analogen“ Signals.  

Die Abtasteffekte lassen sich am einfachsten im Frequenzbereich beobachten.  
Zur Simulation der Fourier-Transformation eines zeitkontinuierlichen Signals wird die Funktion `lafplot()` verwendet  
(siehe `lafplot.m` für weitere Hilfe).

## a) Simulation des folgenden Signals

$$
s(t) = \cos(2\pi f_0 t +\phi), \quad 0 \leq t \leq T
$$

Die Signalfrequenz soll $f_0 = 2\ \mathrm{kHz}$ betragen und die Abtastfrequenz  
$\frac{1}{\Delta t} = 80\ \mathrm{kHz}$.  
Wählen Sie $T$ so, dass das Signal ungefähr 2000 Abtastwerte des simulierten analogen Signals enthält.  

Stellen Sie das „analoge“ Signal mit der Funktion `plot()` dar.  
Achten Sie dabei auf die korrekte Skalierung der Zeitachse.

In [ ]:
# BEGIN SOLUTION
fsim = 80000;
delta_t = (0:1/fsim:0.02)'; 
f0 = 2000;
s  = cos(2*pi*f0*delta_t);

% # grafische Ausgabe # %
figure; 
plot(delta_t,s); 
xlabel('Zeit (sek)'); 
ylabel('Amplitude');
title('Analoges Signal');
# END SOLUTION

## b) Spektrum des „analogen“ Signals darstellen

Verwenden Sie die Funktion `lafplot()`, um das Spektrum darzustellen.

In [ ]:
# BEGIN SOLUTION
fsim = 80000;
delta_t = (0:1/fsim:0.02)'; 
f0 = 2000;
s  = cos(2*pi*f0*delta_t);

% # grafische Ausgabe # %
figure; 
lafplot(s,1/80000);
# END SOLUTION

## c) Abtastung mit einem A/D-Wandler

Ein A/D-Wandler tastet ein analoges Signal mit der Periode $T_A$ ab.  
Simulieren Sie dieses Verhalten, indem Sie das analoge Signal mit einer Frequenz von $f_A = 8\ \mathrm{kHz}$ abtasten und das Ergebnis darstellen.

In [ ]:
# BEGIN SOLUTION
fsim = 80000;
delta_t = (0:1/fsim:0.02)';
f0 = 2000;
s  = cos(2*pi*f0*delta_t);
fa = 8000;
l  = fsim/fa;
index    = (1:l:length(s))-1;
index(1) = [];
sd = s(index);

% # grafische Ausgabe # %
figure; 
stem(sd(1:50)); 
grid; 
xlabel('k'); 
ylabel('Amplitude');
title('Diskretes Signal');
# END SOLUTION

## d) Spektrum des abgetasteten Signals berechnen

Verwenden Sie die Funktion `ldtft()` (siehe `ldtft.m`).

In [ ]:
# BEGIN SOLUTION
fsim = 80000;
delta_t = (0:1/fsim:0.02)';   
f0 = 2000;
s  = cos(2*pi*f0*delta_t);
fa = 8000;
l  = fsim/fa;
index    = (1:l:length(s))-1;
index(1) = [];
sd = s(index);
[H,W] = ldtft(sd,512);

% # grafische Ausgabe # %
figure; 
plot(W/pi,abs(H)); 
grid; 
xlabel('Omega/pi'); 
ylabel('Amplitude');
title('Zeitdiskrete Fourier-Transformation (Betrag)');
# END SOLUTION

## e) Entwurf eines Tiefpassfilters zur Rekonstruktion des analogen Signals

Dieser analoge Rekonstruktions-Tiefpassfilter soll erneut simuliert werden.  
Verwenden Sie die MATLAB-Funktion:

```
[b,a] = cheby2(9, 60, fcut)
```

Diese Funktion entwirft einen Tiefpass 9. Ordnung mit einer Sperrbanddämpfung von $60\ \mathrm{dB}$.  
Die analoge Grenzfrequenz `fcut` soll $\frac{1}{2}f_A$ betragen.  
In MATLAB muss diese Frequenz normalisiert werden:  

```
fcut = fsamp / fsim
```

Stellen Sie die Übertragungsfunktion des Tiefpassfilters mit der Funktion `freqz()` dar.

In [ ]:
# BEGIN SOLUTION
fa    = 8000;
fsim  = 80000;
fcut  = fa/fsim;
[b,a] = cheby2(9,60,fcut);

[H,W] = freqz(b,a,128,fsim);

% # grafische Ausgabe # %
% Frequenzgang
figure; 
semilogy(W,abs(H)); 
grid; 
xlabel('Frequenz in Hz'); 
ylabel('Betrag'); 
title('Rekonstruktions-Tschebyscheff-TP');

% Phasengang
figure; 
plot(W,angle(H));   
grid; 
xlabel('Frequenz in Hz'); 
ylabel('Phase');
title('Rekonstruktions-Tschebyscheff-TP');
# END SOLUTION

## f) Rekonstruktion mit D/A-Wandler

Ein D/A-Wandler erzeugt ein treppenförmiges analoges Zeitsignal.  
Dieses Signal wird durch den entworfenen Chebyshev-Tiefpassfilter geglättet.  
Um diesen Prozess zu simulieren, werden Nullen zwischen die Abtastwerte eingefügt und das Signal anschließend mit dem entworfenen Rekonstruktions-Tiefpassfilter gefiltert.  

Stellen Sie das rekonstruierte analoge Signal sowohl im Zeit- als auch im Frequenzbereich dar.  

**MATLAB-Hinweis:** Die Funktion `lzerofill()` fügt die gewünschte Anzahl von Nullen zwischen die Elemente eines Vektors ein.

In [ ]:
# BEGIN SOLUTION
fsim = 80000;             % analoges Signal
delta_t = (0:1/fsim:0.02)';   
f0 = 2000;
s  = cos(2*pi*f0*delta_t);
fa = 8000;                % Abtastung
l  = fsim/fa;
index = (1:l:length(s))-1;
index(1) = [];
sd = s(index);            % diskretes Signal
sa = lzerofill(sd,l);     % Rekonstruktion/Einfuegen von Nullen 
fsim = 80000;             % Tiefpass entwerfen
fcut = fa/fsim;           % siehe Loesung 
[b,a]=cheby2(9,60,fcut);
sr = filter(b,a,sa);      % Rekonstruktion

% # grafische Ausgabe # %
% rekonstruierter Sinus
figure; 
plot(sr); 
title('Rekonstruierter Sinus'); 
xlabel('Zeit in Abtastwerten'); 
ylabel('Amplitude'); 

% Amplitude
figure; 
lafplot(sr,1/80000); 
ylabel('Amplitude');
# END SOLUTION